In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("modified_freelancer.csv")

In [3]:
# Verify the dataset
print("Dataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

# Step 2: Preprocess the Data
# Combine bio and skills into a single input
df['input_text'] = df['bio'] + ' Skills: ' + df['skills']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   skills  10000 non-null  object
 1   bio     10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB
None

First few rows:
                                         skills  \
0  color palette, adobe lightroom, print design   
1                                sql, analytics   
2      data mining, statistics, data processing   
3   sql programming, database architecture, sql   
4      responsive design, user interface design   

                                                 bio  
0  Senior Graphic & Visual Design professional wi...  
1  Mid-level Data Science professional with 4 yea...  
2  Entry-level Data Science professional with 2 y...  
3  Senior Database Management professional with 1...  
4  Senior UI/UX Design professional with 10 years...  


In [4]:
# Preprocessing
max_vocab_size = 10000  # Increased vocab size for 10k rows
max_len = int(np.percentile([len(x.split()) for x in df['input_text']], 95))  # Dynamic max length


In [5]:
# Tokenize the text
tokenizer = Tokenizer(num_words=max_vocab_size, lower=True, oov_token='<OOV>')
tokenizer.fit_on_texts(df['input_text'])
sequences = tokenizer.texts_to_sequences(df['input_text'])
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')


In [6]:
# Convert to numpy array
X = np.array(padded_sequences)

In [7]:
# Split into training and validation sets
X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)


In [8]:
# Step 3: Define and Build the Autoencoder Model
def build_autoencoder(input_dim, max_len, embedding_dim=64, latent_dim=32):
    model = Sequential([
        # Encoder
        Embedding(input_dim=input_dim, output_dim=embedding_dim, input_length=max_len),
        LSTM(128, activation='relu', return_sequences=False),
        Dense(latent_dim, activation='relu'),  # Latent space bottleneck
        
        # Decoder
        RepeatVector(max_len),
        LSTM(128, activation='relu', return_sequences=True),
        TimeDistributed(Dense(input_dim, activation='softmax'))
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy')
    return model


In [9]:
# Build the model
autoencoder = build_autoencoder(input_dim=max_vocab_size, max_len=max_len)
autoencoder.summary()

C:\Users\Siddhesh Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector (RepeatVector)         │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)